# Indonesian Batik Dataset v2.0 — Leak-Free Starter Notebook

This starter notebook accompanies **[Indonesian Batik Dataset (Enhanced & Cleaned) v2.0](https://www.kaggle.com/datasets/fisheightcharacter/indonesian-batik-dataset-enhanced-and-cleaned)** (`3,470` full-resolution images across `28` motif classes, `3,140` DINOv2 semantic clusters, plus `177` OOD calibration negatives).

### Key Best Practices Demonstrated
1. **Zero-Leakage Splitting:** Uses `StratifiedGroupKFold` on `metadata.csv["cluster_id"]` (`cos >= 0.88`) instead of random `train_test_split`.
2. **Orientation-Safe Augmentations:** Disables horizontal flips for `Solo_Parang`, `Yogyakarta_Parang`, and `Sulawesi_Selatan_Lontara`, and caps hue jitter (`<= 5`) to preserve *Sogan* and *Lasem* dye palettes.
3. **Class-Balanced 2-Phase Fine-Tuning:** Uses capped Effective Number of Samples weights (`<= 3.0`) with PyTorch + `timm` (switchable between `DINOv2-ViT-S/14`, `ConvNeXt-Tiny`, and `EfficientNetV2-S`).
4. **Calibration & OOD Evaluation:** Fits post-hoc Temperature Scaling ($T$) and measures Out-of-Distribution AUROC against `ood_negatives/` (including Indonesian *Songket*, *Tenun*, and *Ulos*).

In [ ]:
# ============================================================
# Indonesian Batik v2.0 — Leak-Free Starter (PyTorch + timm)
# ============================================================
import os, random
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset

# --- Configuration ---
DATA_ROOT   = Path("/kaggle/input/indonesian-batik-dataset-enhanced-and-cleaned")
BACKBONE    = "vit_small_patch14_dinov2.lvd142m"  # or "convnext_tiny.fb_in22k_ft_in1k_384" / "tf_efficientnetv2_s.in21k_ft_in1k"
IMG_SIZE    = 224  # 224 for DINOv2-ViT-S/14, 288 for ConvNeXt-Tiny, 300 for EfficientNetV2-S
BATCH_SIZE  = 32
P1_EPOCHS   = 5
P2_EPOCHS   = 15
SEED        = 42
ORIENTATION_SENSITIVE = {"Solo_Parang", "Yogyakarta_Parang", "Sulawesi_Selatan_Lontara"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | Backbone: {BACKBONE} ({IMG_SIZE}x{IMG_SIZE})")

In [ ]:
# 1. Load metadata.csv & OOD calibration manifest
df = pd.read_csv(DATA_ROOT / "metadata.csv")
df["abs_path"] = df["rel_path"].apply(lambda r: str(DATA_ROOT / r))

ood_df = pd.read_csv(DATA_ROOT / "ood_negatives" / "ood_manifest.csv")
ood_df["abs_path"] = ood_df["rel_path"].apply(lambda r: str(DATA_ROOT / r))

class_names = sorted(df["label"].unique())
lbl2idx = {c: i for i, c in enumerate(class_names)}
df["label_idx"] = df["label"].map(lbl2idx)
NUM_CLASSES = len(class_names)

print(f"Batik images : {len(df)} | Classes: {NUM_CLASSES} | Semantic clusters: {df['cluster_id'].nunique()}")
print(f"OOD negatives: {len(ood_df)} across {ood_df['bucket'].nunique()} buckets")
print(df.groupby("label").size().sort_values().to_string())

In [ ]:
# 2. Visualize one sample per class (28 classes)
cols = 7
rows = (NUM_CLASSES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 2.8))
axes = axes.flatten()
for i, cls in enumerate(class_names):
    p = df[df["label"] == cls].sample(1, random_state=SEED).iloc[0]["abs_path"]
    im = ImageOps.exif_transpose(Image.open(p)).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    axes[i].imshow(im)
    axes[i].set_title(cls, fontsize=7)
    axes[i].axis("off")
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Strictly Leak-Free Split by cluster_id (StratifiedGroupKFold)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf.split(df, df["label_idx"], groups=df["cluster_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

shared = set(train_df["cluster_id"]) & set(val_df["cluster_id"])
assert len(shared) == 0, f"Leakage detected: {len(shared)} shared clusters!"
print(f"Leak-free split verified -> train={len(train_df)}, val={len(val_df)}, shared_clusters=0")

In [ ]:
# 4. Orientation-Safe Albumentations Dataset & DataLoaders
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

class BatikDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, is_train: bool):
        self.df = frame
        self.is_train = is_train
        if is_train:
            self.tfm_flip = A.Compose([
                A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.68, 1.0), p=1.0),
                A.HorizontalFlip(p=0.5),
                A.Affine(scale=(0.92, 1.08), rotate=(-20, 20), border_mode=cv2.BORDER_REFLECT_101, p=0.6),
                A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=18, val_shift_limit=18, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18, p=0.5),
            ])
            self.tfm_noflip = A.Compose([
                A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.72, 1.0), p=1.0),
                A.Affine(scale=(0.94, 1.06), rotate=(-12, 12), border_mode=cv2.BORDER_REFLECT_101, p=0.6),
                A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=18, val_shift_limit=18, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18, p=0.5),
            ])
        else:
            self.tfm_eval = A.Compose([
                A.SmallestMaxSize(max_size=IMG_SIZE),
                A.CenterCrop(height=IMG_SIZE, width=IMG_SIZE),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = np.asarray(ImageOps.exif_transpose(Image.open(row["abs_path"])).convert("RGB"), dtype=np.uint8)
        if self.is_train:
            tfm = self.tfm_noflip if row["label"] in ORIENTATION_SENSITIVE else self.tfm_flip
            arr = tfm(image=arr)["image"]
        else:
            arr = self.tfm_eval(image=arr)["image"]
        x = (arr.astype(np.float32).transpose(2, 0, 1) / 255.0 - MEAN) / STD
        y = int(row["label_idx"]) if "label_idx" in self.df.columns else -1
        return torch.from_numpy(x), y

train_loader = DataLoader(BatikDataset(train_df, True),  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(BatikDataset(val_df,   False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
ood_loader   = DataLoader(BatikDataset(ood_df,   False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# 5. Model, Effective-Number Class Weights & 2-Phase Fine-Tuning
counts = np.array([(train_df["label_idx"] == c).sum() for c in range(NUM_CLASSES)], dtype=np.float64)
eff_w  = (1.0 - 0.999) / (1.0 - np.power(0.999, np.maximum(counts, 1.0)))
eff_w  = np.clip(eff_w / eff_w.mean(), 0.25, 3.0).astype(np.float32)

kwargs = {"pretrained": True, "num_classes": 0}
if "vit_" in BACKBONE or "dinov2" in BACKBONE:
    kwargs["img_size"] = IMG_SIZE

class BatikClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, **kwargs)
        self.head = nn.Sequential(
            nn.LayerNorm(self.backbone.num_features),
            nn.Dropout(0.35),
            nn.Linear(self.backbone.num_features, NUM_CLASSES),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = BatikClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=torch.from_numpy(eff_w).to(DEVICE), label_smoothing=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

@torch.no_grad()
def predict(loader):
    model.eval()
    logits_all, y_all = [], []
    for xb, yb in loader:
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits_all.append(model(xb.to(DEVICE)).float().cpu().numpy())
        y_all.append(yb.numpy())
    return np.concatenate(logits_all), np.concatenate(y_all)

# Phase 1: Frozen backbone warmup
for p in model.backbone.parameters():
    p.requires_grad = False
opt = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
best_f1, best_state = -1.0, None

for ep in range(1, P1_EPOCHS + 1):
    model.train()
    model.backbone.eval()
    for xb, yb in train_loader:
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    vl, vy = predict(val_loader)
    f1 = f1_score(vy, vl.argmax(1), average="macro")
    if f1 > best_f1:
        best_f1, best_state = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"[P1 {ep:02d}/{P1_EPOCHS}] val_acc={accuracy_score(vy, vl.argmax(1))*100:.2f}% | val_macro_f1={f1*100:.2f}%")

# Phase 2: Unfreeze top blocks with discriminative learning rates
if hasattr(model.backbone, "blocks"):
    for blk in list(model.backbone.blocks)[-6:]:
        for p in blk.parameters():
            p.requires_grad = True
    for attr in ("norm", "fc_norm"):
        m = getattr(model.backbone, attr, None)
        if m is not None:
            for p in m.parameters():
                p.requires_grad = True
else:
    for p in model.backbone.parameters():
        p.requires_grad = True

opt2 = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 2.5e-5, "weight_decay": 0.05},
    {"params": model.head.parameters(), "lr": 5e-4, "weight_decay": 0.02},
])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=P2_EPOCHS * len(train_loader))

for ep in range(1, P2_EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        opt2.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
            loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt2)
        scaler.update()
        sched.step()
    vl, vy = predict(val_loader)
    f1 = f1_score(vy, vl.argmax(1), average="macro")
    tag = ""
    if f1 > best_f1:
        best_f1, best_state, tag = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}, " *BEST*"
    print(f"[P2 {ep:02d}/{P2_EPOCHS}] val_acc={accuracy_score(vy, vl.argmax(1))*100:.2f}% | val_macro_f1={f1*100:.2f}%{tag}")

In [ ]:
# 6. Final Leak-Free Evaluation, Temperature Scaling & OOD AUROC
model.load_state_dict(best_state)
val_logits, val_y = predict(val_loader)
ood_logits, _     = predict(ood_loader)

def softmax_t(z, T=1.0):
    s = z / T
    s = s - s.max(axis=1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=1, keepdims=True)

# Fit scalar Temperature T minimizing validation NLL
grid = np.geomspace(0.2, 5.0, 60)
best_T = float(grid[np.argmin([
    -np.mean(np.log(np.clip(softmax_t(val_logits, t)[np.arange(len(val_y)), val_y], 1e-12, 1.0)))
    for t in grid
])])

val_probs = softmax_t(val_logits, best_T)
ood_probs = softmax_t(ood_logits, best_T)

top1 = accuracy_score(val_y, val_probs.argmax(1))
top3 = np.mean([val_y[i] in np.argsort(-val_probs[i])[:3] for i in range(len(val_y))])
mf1  = f1_score(val_y, val_probs.argmax(1), average="macro")

y_ood_bin = np.concatenate([np.zeros(len(val_probs)), np.ones(len(ood_probs))])
ood_auroc = roc_auc_score(y_ood_bin, np.concatenate([1.0 - val_probs.max(1), 1.0 - ood_probs.max(1)]))

print(f"Leak-Free Val Top-1 Acc : {top1*100:.2f}%")
print(f"Leak-Free Val Top-3 Acc : {top3*100:.2f}%")
print(f"Leak-Free Val Macro-F1  : {mf1*100:.2f}%")
print(f"Optimal Temperature T   : {best_T:.3f}")
print(f"OOD AUROC (vs 177 negs) : {ood_auroc:.4f}")
print("\n" + classification_report(val_y, val_probs.argmax(1), target_names=class_names, digits=4))